In [4]:
!uv add langchain langchain_community langchain-openai langchainhub
!uv add tiktoken chromadb loguru python-dotenv

Resolved 233 packages in 2ms
Audited 222 packages in 0.13ms
Resolved 233 packages in 2ms
Audited 222 packages in 0.13ms


In [ ]:
import os
import dotenv
from flashboot_core.utils import project_utils

root_path = project_utils.get_root_path()
dotenv.load_dotenv(root_path / ".env", override=True)

True

In [2]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from loguru import logger

In [3]:
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]

raw_documents = [
    "产品 A 尺码偏大，建议购买时选择小一码以获得更好的贴合感。",
    "产品 B 尺码偏小，内部空间较为紧凑，建议选购大一码。",
    "产品 C 尺码适中，穿着舒适。",
]

docs = [
    Document(page_content=x, metadata={"source": "size_guide"}) for x in raw_documents
]
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
splits = text_splitter.split_documents(docs)

embeddings = OpenAIEmbeddings(
    model="text-embedding-v2",
    openai_api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
    check_embedding_ctx_length=False,
)

vectorstore = Chroma.from_documents(splits, embeddings)
logger.info("Vector store created.")

query = "哪件产品尺码偏大？"
retrieved_docs = vectorstore.similarity_search(query)
context_text = "\n".join([doc.page_content for doc in retrieved_docs])

llm = ChatOpenAI(
    model="deepseek-v3.2",
    openai_api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

template = "Answer the question base only on the following context:\n{context}\n\nQuestion: {question}"
prompt = ChatPromptTemplate.from_template(template)
chain = prompt | llm

response = chain.invoke({"context": context_text, "question": query})
print(f"\nAI Response:\n{response.content}")

2026-01-26 13:37:52.389 | INFO     | __main__:<module>:24 - Vector store created.



AI Response:
根据上下文，产品 A 尺码偏大。
